# 04 — Model Comparison: Which Algorithm Wins?

**Goal:** Compare 4 different ML algorithms on the same sentiment data to find the best performer.

**Models we'll test:**
1. **Multinomial Naive Bayes** — probabilistic classifier built for text data
2. **Linear SVM (Support Vector Machine)** — finds the optimal decision boundary
3. **Random Forest** — ensemble of decision trees
4. **Logistic Regression** — our Phase 3 baseline (91.4% accuracy)

**What we'll measure:**
- Accuracy, Precision, Recall, F1 Score
- Training time
- Performance on custom reviews

**Key question:** Can we beat the Logistic Regression baseline?

---
## Part 1: Setup

**YOUR TASK:** Import all required libraries.

New imports for this notebook:
- `from sklearn.naive_bayes import MultinomialNB` — Naive Bayes classifier
- `from sklearn.svm import LinearSVC` — Linear Support Vector Classifier
- `from sklearn.ensemble import RandomForestClassifier` — Random Forest
- `import time` — to measure training speed
- `import numpy as np` — numerical operations

In [ ]:
# YOUR CODE: Import all libraries
import pandas as pd
import numpy as np
import time
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer

# Our 4 models
from sklearn.naive_bayes import MultinomialNB
from sklearn.svm import LinearSVC
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression

# Evaluation metrics
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix, f1_score

import warnings
warnings.filterwarnings('ignore')

print("All libraries imported!")

---
## Part 2: Load & Prepare Data

Same dataset as Phase 3 — 238K+ multi-domain reviews.

**YOUR TASK:** Load the cleaned data and prepare X/y.

In [ ]:
# YOUR CODE: Load the cleaned dataset
df = pd.read_csv('../data/processed/combined_reviews_clean.csv')
df = df.dropna(subset=['clean_review', 'sentiment'])
df = df[df['clean_review'].str.strip() != '']

print(f"Dataset: {df.shape[0]:,} reviews")
print(f"\nSentiment distribution:")
print(df['sentiment'].value_counts())
print(f"\nSources: {df['source'].unique()}")

---
## Part 3: Train/Test Split & TF-IDF

**Important:** We use the EXACT same split parameters as Phase 3 (`random_state=42`, `test_size=0.2`).

This ensures a fair comparison — every model sees the same training and test data.

In [ ]:
# YOUR CODE: Split and vectorize
X = df['clean_review']
y = df['sentiment']

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

tfidf = TfidfVectorizer(max_features=50000)
X_train_tfidf = tfidf.fit_transform(X_train)
X_test_tfidf = tfidf.transform(X_test)

print(f"Training set: {X_train_tfidf.shape[0]:,} reviews")
print(f"Test set:     {X_test_tfidf.shape[0]:,} reviews")
print(f"Features:     {X_train_tfidf.shape[1]:,} TF-IDF features")

---
## Part 4: Helper Function

To avoid repeating code, we'll create a helper that trains a model, times it, predicts, and stores the results.

**NEW CONCEPT: DRY (Don't Repeat Yourself)**

When you need to do the same thing 4 times, write a function.

In [ ]:
# Storage for all results
all_results = []
all_predictions = {}
all_models = {}

def train_and_evaluate(name, model, X_train, y_train, X_test, y_test):
    """Train a model, time it, evaluate it, and store the results."""
    # Train with timing
    start = time.time()
    model.fit(X_train, y_train)
    train_time = time.time() - start
    
    # Predict
    y_pred = model.predict(X_test)
    
    # Metrics
    acc = accuracy_score(y_test, y_pred)
    f1_neg = f1_score(y_test, y_pred, pos_label='negative')
    f1_pos = f1_score(y_test, y_pred, pos_label='positive')
    
    # Store results
    all_results.append({
        'Model': name,
        'Accuracy': acc,
        'F1 (Negative)': f1_neg,
        'F1 (Positive)': f1_pos,
        'Training Time (s)': train_time
    })
    all_predictions[name] = y_pred
    all_models[name] = model
    
    return acc, train_time, y_pred

print("Helper function ready!")

---
## Model 1: Multinomial Naive Bayes

**How it works:** Calculates the probability of each class given the words in the review, using Bayes' theorem.

**Why try it?**
- Specifically designed for text classification
- Extremely fast to train
- Works well with TF-IDF features
- Often a strong baseline for NLP tasks

**Limitation:** Assumes features are independent (the "naive" assumption) — in reality, words are correlated.

In [ ]:
# YOUR CODE: Train Multinomial Naive Bayes
nb_acc, nb_time, nb_pred = train_and_evaluate(
    'Naive Bayes', MultinomialNB(),
    X_train_tfidf, y_train, X_test_tfidf, y_test
)

print(f"Naive Bayes trained in {nb_time:.2f} seconds")
print(f"Accuracy: {nb_acc:.4f} ({nb_acc*100:.1f}%)")
print(f"\n{classification_report(y_test, nb_pred)}")

In [ ]:
# Confusion Matrix — Naive Bayes
cm_nb = confusion_matrix(y_test, nb_pred)
plt.figure(figsize=(6, 4))
sns.heatmap(cm_nb, annot=True, fmt='d', cmap='Blues',
            xticklabels=['negative', 'positive'],
            yticklabels=['negative', 'positive'])
plt.title('Naive Bayes — Confusion Matrix', fontsize=14)
plt.xlabel('Predicted')
plt.ylabel('Actual')
plt.tight_layout()
plt.show()

---
## Model 2: Linear SVM (Support Vector Machine)

**How it works:** Finds the hyperplane that best separates positive and negative reviews in the feature space, maximizing the margin between classes.

**Why try it?**
- Excellent for high-dimensional data (50,000 TF-IDF features)
- Often the top performer for text classification
- Efficient with sparse matrices

**Key parameter:** `max_iter=1000` — allows enough iterations to converge.

In [ ]:
# YOUR CODE: Train Linear SVM
svm_acc, svm_time, svm_pred = train_and_evaluate(
    'Linear SVM', LinearSVC(max_iter=1000),
    X_train_tfidf, y_train, X_test_tfidf, y_test
)

print(f"Linear SVM trained in {svm_time:.2f} seconds")
print(f"Accuracy: {svm_acc:.4f} ({svm_acc*100:.1f}%)")
print(f"\n{classification_report(y_test, svm_pred)}")

In [ ]:
# Confusion Matrix — Linear SVM
cm_svm = confusion_matrix(y_test, svm_pred)
plt.figure(figsize=(6, 4))
sns.heatmap(cm_svm, annot=True, fmt='d', cmap='Greens',
            xticklabels=['negative', 'positive'],
            yticklabels=['negative', 'positive'])
plt.title('Linear SVM — Confusion Matrix', fontsize=14)
plt.xlabel('Predicted')
plt.ylabel('Actual')
plt.tight_layout()
plt.show()

---
## Model 3: Random Forest

**How it works:** Builds 100 different decision trees, each trained on a random subset of the data. Final prediction = majority vote.

**Why try it?**
- Ensemble method — combines many weak learners into a strong one
- Resistant to overfitting
- Can capture non-linear relationships

**Key parameters:**
- `n_estimators=100` — number of trees
- `n_jobs=-1` — use all CPU cores for parallel training
- `random_state=42` — reproducibility

**Warning:** Random Forest is typically slower than linear models on high-dimensional sparse data like TF-IDF.

In [ ]:
# YOUR CODE: Train Random Forest
rf_acc, rf_time, rf_pred = train_and_evaluate(
    'Random Forest', RandomForestClassifier(n_estimators=100, random_state=42, n_jobs=-1),
    X_train_tfidf, y_train, X_test_tfidf, y_test
)

print(f"Random Forest trained in {rf_time:.2f} seconds")
print(f"Accuracy: {rf_acc:.4f} ({rf_acc*100:.1f}%)")
print(f"\n{classification_report(y_test, rf_pred)}")

In [ ]:
# Confusion Matrix — Random Forest
cm_rf = confusion_matrix(y_test, rf_pred)
plt.figure(figsize=(6, 4))
sns.heatmap(cm_rf, annot=True, fmt='d', cmap='Oranges',
            xticklabels=['negative', 'positive'],
            yticklabels=['negative', 'positive'])
plt.title('Random Forest — Confusion Matrix', fontsize=14)
plt.xlabel('Predicted')
plt.ylabel('Actual')
plt.tight_layout()
plt.show()

---
## Model 4: Logistic Regression (Our Phase 3 Baseline)

**How it works:** Fits a linear boundary using the logistic (sigmoid) function to estimate class probabilities.

We already know this achieves 91.4% accuracy from Phase 3. Let's re-train it here so all models are compared under identical conditions.

In [ ]:
# YOUR CODE: Train Logistic Regression (baseline)
lr_acc, lr_time, lr_pred = train_and_evaluate(
    'Logistic Regression', LogisticRegression(max_iter=1000),
    X_train_tfidf, y_train, X_test_tfidf, y_test
)

print(f"Logistic Regression trained in {lr_time:.2f} seconds")
print(f"Accuracy: {lr_acc:.4f} ({lr_acc*100:.1f}%)")
print(f"\n{classification_report(y_test, lr_pred)}")

In [ ]:
# Confusion Matrix — Logistic Regression
cm_lr = confusion_matrix(y_test, lr_pred)
plt.figure(figsize=(6, 4))
sns.heatmap(cm_lr, annot=True, fmt='d', cmap='Purples',
            xticklabels=['negative', 'positive'],
            yticklabels=['negative', 'positive'])
plt.title('Logistic Regression — Confusion Matrix', fontsize=14)
plt.xlabel('Predicted')
plt.ylabel('Actual')
plt.tight_layout()
plt.show()

---
## Part 5: Head-to-Head Comparison

Now the moment of truth — which model wins?

In [ ]:
# Build comparison table
results = pd.DataFrame(all_results)
results = results.sort_values('Accuracy', ascending=False).reset_index(drop=True)

print("=" * 75)
print("                    MODEL COMPARISON RESULTS")
print("=" * 75)
print(results.to_string(index=False, float_format=lambda x: f'{x:.4f}'))
print("=" * 75)

best = results.iloc[0]
baseline_acc = results[results['Model'] == 'Logistic Regression']['Accuracy'].values[0]

print(f"\nBest Model: {best['Model']}")
print(f"Best Accuracy: {best['Accuracy']:.4f} ({best['Accuracy']*100:.1f}%)")

if best['Accuracy'] > baseline_acc:
    improvement = (best['Accuracy'] - baseline_acc) * 100
    print(f"Improvement over baseline: +{improvement:.2f} percentage points")
elif best['Model'] == 'Logistic Regression':
    print("The baseline Logistic Regression is still the best!")
else:
    print("No model beat the Logistic Regression baseline.")

In [ ]:
# Visual comparison — Accuracy & Training Time
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

colors = ['#2196F3', '#4CAF50', '#FF9800', '#9C27B0']

# Accuracy bar chart
bars1 = axes[0].bar(results['Model'], results['Accuracy'], color=colors[:len(results)])
axes[0].set_ylim(min(results['Accuracy']) - 0.03, max(results['Accuracy']) + 0.02)
axes[0].set_ylabel('Accuracy', fontsize=12)
axes[0].set_title('Model Accuracy Comparison', fontsize=14, fontweight='bold')
axes[0].tick_params(axis='x', rotation=15)
for bar, acc in zip(bars1, results['Accuracy']):
    axes[0].text(bar.get_x() + bar.get_width()/2., bar.get_height() + 0.002,
                f'{acc:.3f}', ha='center', va='bottom', fontweight='bold', fontsize=11)

# Training time bar chart
bars2 = axes[1].bar(results['Model'], results['Training Time (s)'], color=colors[:len(results)])
axes[1].set_ylabel('Training Time (seconds)', fontsize=12)
axes[1].set_title('Training Time Comparison', fontsize=14, fontweight='bold')
axes[1].tick_params(axis='x', rotation=15)
for bar, t in zip(bars2, results['Training Time (s)']):
    axes[1].text(bar.get_x() + bar.get_width()/2., bar.get_height() + max(results['Training Time (s)'])*0.02,
                f'{t:.1f}s', ha='center', va='bottom', fontweight='bold', fontsize=11)

plt.tight_layout()
plt.savefig('../data/processed/model_comparison.png', dpi=150, bbox_inches='tight')
plt.show()
print("Chart saved to data/processed/model_comparison.png")

In [ ]:
# F1 Score comparison — Negative vs Positive for each model
fig, ax = plt.subplots(figsize=(10, 5))

x = np.arange(len(results))
width = 0.35

bars_neg = ax.bar(x - width/2, results['F1 (Negative)'], width, label='F1 (Negative)', color='#EF5350', alpha=0.85)
bars_pos = ax.bar(x + width/2, results['F1 (Positive)'], width, label='F1 (Positive)', color='#66BB6A', alpha=0.85)

ax.set_ylabel('F1 Score', fontsize=12)
ax.set_title('F1 Score by Class — All Models', fontsize=14, fontweight='bold')
ax.set_xticks(x)
ax.set_xticklabels(results['Model'], rotation=15)
ax.legend(fontsize=11)
ax.set_ylim(0.80, 0.96)

for bar in bars_neg:
    ax.text(bar.get_x() + bar.get_width()/2., bar.get_height() + 0.003,
            f'{bar.get_height():.3f}', ha='center', va='bottom', fontsize=9)
for bar in bars_pos:
    ax.text(bar.get_x() + bar.get_width()/2., bar.get_height() + 0.003,
            f'{bar.get_height():.3f}', ha='center', va='bottom', fontsize=9)

plt.tight_layout()
plt.savefig('../data/processed/f1_comparison.png', dpi=150, bbox_inches='tight')
plt.show()
print("Chart saved to data/processed/f1_comparison.png")

---
## Part 6: Test All Models on Custom Reviews

Let's see how each model handles the same set of test reviews — including tricky ones like sarcasm and neutral text.

In [ ]:
# Test all models on custom reviews
test_reviews = [
    "This product is absolutely amazing, I love it!",
    "Terrible experience, worst purchase I ever made.",
    "It was okay, nothing special but not bad either.",
    "The quality is outstanding and the price is fair.",
    "Broke after two days. Total waste of money.",
    "Not what I expected but pleasantly surprised.",
    "The customer service was rude and unhelpful.",
    "Would definitely recommend to friends and family!",
    "Meh. It works I guess.",
    "DO NOT BUY THIS. Absolute scam."
]

test_tfidf = tfidf.transform(test_reviews)

# Get predictions from all models
model_names = list(all_models.keys())
print("=" * 100)
header = f"{'Review':<50}"
for name in model_names:
    short_name = name[:8]
    header += f" {short_name:>10}"
print(header)
print("=" * 100)

for review in test_reviews:
    review_tfidf = tfidf.transform([review])
    short_review = review[:47] + "..." if len(review) > 50 else review
    row = f"{short_review:<50}"
    for name in model_names:
        pred = all_models[name].predict(review_tfidf)[0]
        label = "pos" if pred == "positive" else "neg"
        row += f" {label:>10}"
    print(row)

print("=" * 100)

---
## Part 7: All Confusion Matrices Side by Side

In [ ]:
# All confusion matrices in one figure
fig, axes = plt.subplots(1, 4, figsize=(20, 4))
cmaps = ['Blues', 'Greens', 'Oranges', 'Purples']

for idx, (name, preds) in enumerate(all_predictions.items()):
    cm = confusion_matrix(y_test, preds)
    sns.heatmap(cm, annot=True, fmt='d', cmap=cmaps[idx],
                xticklabels=['neg', 'pos'],
                yticklabels=['neg', 'pos'],
                ax=axes[idx])
    acc = accuracy_score(y_test, preds)
    axes[idx].set_title(f'{name}\n({acc:.1%})', fontsize=12, fontweight='bold')
    axes[idx].set_xlabel('Predicted')
    if idx == 0:
        axes[idx].set_ylabel('Actual')
    else:
        axes[idx].set_ylabel('')

plt.suptitle('Confusion Matrices — All Models', fontsize=16, fontweight='bold', y=1.05)
plt.tight_layout()
plt.savefig('../data/processed/all_confusion_matrices.png', dpi=150, bbox_inches='tight')
plt.show()
print("Chart saved to data/processed/all_confusion_matrices.png")

---
## Part 8: Save the Best Model

In [ ]:
import joblib

# Save the best-performing model
best_name = results.iloc[0]['Model']
best_model = all_models[best_name]

joblib.dump(best_model, '../models/best_model_phase4.pkl')
print(f"Best model ({best_name}) saved to models/best_model_phase4.pkl")

# Also save all models for future use
for name, model in all_models.items():
    filename = name.lower().replace(' ', '_')
    joblib.dump(model, f'../models/{filename}_model.pkl')
    print(f"  Saved: models/{filename}_model.pkl")

print(f"\nAll models saved. TF-IDF vectorizer from Phase 3 is still valid.")

---
## Summary

Fill this in after running the notebook:

1. **Best Model:** _____ with _____% accuracy
2. **Fastest Model:** _____ (_____ seconds)
3. **Beat the baseline?** Yes/No — by _____ percentage points
4. **Most false positives:** _____
5. **Most false negatives:** _____

### Key Takeaways

- **Linear models** (SVM, Logistic Regression) tend to work best for text classification because TF-IDF features are already high-dimensional and linearly separable
- **Naive Bayes** is blazing fast but makes a strong independence assumption that hurts accuracy slightly
- **Random Forest** is powerful for tabular data but less efficient for sparse, high-dimensional text features
- Training time matters in production — a model that's 0.5% less accurate but 10x faster may be the better choice

### Questions to Think About

- If you had to deploy one model to production, which would you choose and why?
- How would these results change with a smaller/larger vocabulary (max_features)?
- Would word embeddings (Word2Vec, GloVe) change the Random Forest performance?
- What's the next step to push accuracy even higher? (Hint: Phase 5)

### Next Phase

**Phase 5: Deep Learning** — Fine-tune a DistilBERT transformer model to see if we can break the 93%+ barrier.